# 8.27 — Multilingual & cross-lingual NLP

Cross-lingual NLP tries to let languages share a geometric workspace without pretending their scripts, data, and grammar are the same. Embeddings support transfer and zero-shot classification, while coverage and sampling policies decide who benefits. Save a copy to Drive to edit.

In [ ]:

import math
import random
import sqlite3
from collections import Counter
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np

random.seed(8025)
np.random.seed(8025)


def sigmoid(z):
    return 1.0 / (1.0 + math.exp(-z))


def softmax(values):
    arr = np.array(values, dtype=float)
    shifted = arr - np.max(arr)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum()


def cosine(a, b):
    a_arr = np.array(a, dtype=float)
    b_arr = np.array(b, dtype=float)
    denom = np.linalg.norm(a_arr) * np.linalg.norm(b_arr)
    if denom == 0:
        return 0.0
    return float(np.dot(a_arr, b_arr) / denom)


def accuracy_score(y_true, y_pred):
    correct = sum(int(a == b) for a, b in zip(y_true, y_pred))
    return correct / max(1, len(y_true))


def precision_recall_f1(y_true, y_pred):
    tp = sum(int(a == 1 and b == 1) for a, b in zip(y_true, y_pred))
    fp = sum(int(a == 0 and b == 1) for a, b in zip(y_true, y_pred))
    fn = sum(int(a == 1 and b == 0) for a, b in zip(y_true, y_pred))
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 2 * precision * recall / max(1e-12, precision + recall)
    return precision, recall, f1

LABEL_EMBS = {
    "positive": np.array([1.0, 0.0, 0.2]),
    "negative": np.array([0.0, 1.0, 0.2]),
}
WORD_EMBS = {
    "good": np.array([0.9, 0.1, 0.2]),
    "great": np.array([1.0, 0.1, 0.1]),
    "bad": np.array([0.1, 0.9, 0.2]),
    "awful": np.array([0.1, 1.0, 0.1]),
    "bueno": np.array([0.88, 0.12, 0.2]),
    "malo": np.array([0.15, 0.9, 0.2]),
    "bon": np.array([0.86, 0.14, 0.25]),
    "mauvais": np.array([0.12, 0.88, 0.25]),
    "nzuri": np.array([0.8, 0.2, 0.5]),
    "mbaya": np.array([0.2, 0.8, 0.5]),
    "جيد": np.array([0.75, 0.25, 0.7]),
    "سيء": np.array([0.25, 0.75, 0.7]),
}
SCRIPT_COVERAGE = {
    "en": 1.0,
    "es": 1.0,
    "fr": 0.95,
    "sw": 0.75,
    "ar": 0.6,
}


def embed_text_multilingual(text, language):
    tokens = text.lower().split()
    vectors = [WORD_EMBS[token] for token in tokens if token in WORD_EMBS]
    if not vectors:
        return np.array([0.3, 0.3, 0.8])
    base = np.mean(vectors, axis=0)
    coverage = SCRIPT_COVERAGE.get(language, 0.7)
    noise_axis = np.array([0.0, 0.0, 1.0])
    return coverage * base + (1.0 - coverage) * noise_axis


def crosslingual_classify(emb, label_embs=LABEL_EMBS, losses=None):
    scores = {label: cosine(emb, vec) for label, vec in label_embs.items()}
    pred = max(scores, key=scores.get)
    weights = None
    if losses is not None:
        inv = np.array([1.0 / loss for loss in losses], dtype=float)
        weights = inv / inv.sum()
    return pred, scores, weights


def crosslingual_dataset_ladder():
    d1 = [
        {"language": "en", "text": "good", "label": "positive"},
        {"language": "es", "text": "bueno", "label": "positive"},
        {"language": "en", "text": "bad", "label": "negative"},
        {"language": "es", "text": "malo", "label": "negative"},
    ]
    d2 = d1 + [
        {"language": "fr", "text": "bon", "label": "positive"},
    ]
    d3 = d2 + [
        {"language": "fr", "text": "mauvais", "label": "negative"},
        {"language": "ar", "text": "جيد", "label": "positive"},
        {"language": "ar", "text": "سيء", "label": "negative"},
    ]
    d4 = d3 + [
        {"language": "sw", "text": "nzuri", "label": "positive"},
        {"language": "sw", "text": "mbaya", "label": "negative"},
        {"language": "es", "text": "bueno great", "label": "positive"},
        {"language": "fr", "text": "mauvais bad", "label": "negative"},
    ]
    d5 = d4 + [
        {"language": "sw", "text": "nzuri mbaya", "label": "positive"},
        {"language": "ar", "text": "جيد سيء", "label": "negative"},
        {"language": "en", "text": "great good", "label": "positive"},
        {"language": "es", "text": "malo awful", "label": "negative"},
        {"language": "sw", "text": "mbaya bad", "label": "negative"},
    ]
    return [
        {"rung": "D1", "name": "lesson aligned-vector toy", "items": d1},
        {"rung": "D2", "name": "five clean translation pairs", "items": d2},
        {"rung": "D3", "name": "script coverage and domain mismatch", "items": d3},
        {"rung": "D4", "name": "inline multilingual sentiment set", "items": d4},
        {"rung": "D5", "name": "low-resource noisy examples", "items": d5},
    ]


## The concept, built once: shared geometry and language sampling

The lesson compares multilingual embeddings with cosine and computes sampling weights:

$$\cos(e_x,e_z)=rac{e_x^	op e_z}{\|e_x\|\|e_z\|},\qquad q_\ell=rac{1/L_\ell}{\sum_j 1/L_j}.$$

We implement a classifier over label embeddings and assert the exact lesson weights `[0.625,0.25,0.125]`.

In [ ]:

emb = np.array([0.8, 0.2, 0.0])
pred, scores, weights = crosslingual_classify(emb, losses=[0.2, 0.5, 1.0])

assert pred == "positive"
assert [round(w, 3) for w in weights] == [0.625, 0.25, 0.125]

print("label scores", {key: round(value, 3) for key, value in scores.items()})
print("sampling weights", [round(w, 3) for w in weights])


## Alignment, imbalance, script coverage, and zero-shot labels

The lesson's dot-product matrix has translation pairs on the diagonal, English has `20` times Swahili labels, Arabic coverage is `0.4` below Latin, and document `0` wins a zero-shot label ranking.

In [ ]:

english = np.array([[1, 0], [0, 1]], dtype=float)
spanish = np.array([[0.9, 0.1], [0.1, 0.9]], dtype=float)
similarity = english @ spanish.T
ratio = 1000 / 50
coverage_gap = 1.0 - 0.6
label_vec = np.array([1.0, 0.0])
doc_vecs = np.array([[0.8, 0.2], [0.1, 0.9], [0.7, 0.3]])
doc_scores = doc_vecs @ label_vec

assert similarity.tolist() == [[0.9, 0.1], [0.1, 0.9]]
assert ratio == 20
assert abs(coverage_gap - 0.4) < 1e-12
assert int(np.argmax(doc_scores)) == 0

print("alignment matrix", similarity.tolist())
print("resource ratio", ratio)
print("coverage gap", coverage_gap)
print("zero-shot scores", doc_scores.tolist())


## The dataset ladder: D1 to D5 synthetic multilingual text

The ladder grows from aligned translation pairs to noisy low-resource examples with reduced script coverage and mixed sentiment tokens.

In [ ]:

ladder = crosslingual_dataset_ladder()

for rung in ladder:
    languages = Counter(item["language"] for item in rung["items"])
    labels = Counter(item["label"] for item in rung["items"])
    print(rung["rung"], rung["name"], "n=", len(rung["items"]), dict(languages), dict(labels))
    print("sample:", rung["items"][0])


## Run the same cross-lingual classifier across D1-D5

The metric is classification accuracy, with per-language slices retained so the global score cannot hide low-resource failure.

In [ ]:

results = []

for rung in ladder:
    y_true = []
    y_pred = []
    langs = []
    score_rows = []
    for item in rung["items"]:
        emb = embed_text_multilingual(item["text"], item["language"])
        pred, scores, weights = crosslingual_classify(emb)
        y_true.append(item["label"])
        y_pred.append(pred)
        langs.append(item["language"])
        score_rows.append([scores["positive"], scores["negative"]])
    acc = accuracy_score(y_true, y_pred)
    per_language = {}
    for language in sorted(set(langs)):
        mask = [lang == language for lang in langs]
        lang_true = [gold for gold, keep in zip(y_true, mask) if keep]
        lang_pred = [pred for pred, keep in zip(y_pred, mask) if keep]
        per_language[language] = accuracy_score(lang_true, lang_pred)
    results.append({"rung": rung["rung"], "accuracy": acc, "scores": score_rows, "per_language": per_language})

print("rung accuracy per_language")
for row in results:
    print(row["rung"], round(row["accuracy"], 3), row["per_language"])


## Results visualization: alignment panels and metric curve

In [ ]:

fig, axes = plt.subplots(1, 5, figsize=(15, 3))

for ax, rung, row in zip(axes, ladder, results):
    matrix = np.array(row["scores"][:6], dtype=float)
    ax.imshow(matrix, aspect="auto", cmap="magma", vmin=0, vmax=1)
    ax.set_title(rung["rung"])
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["pos", "neg"])
    ax.set_yticks([])

plt.suptitle("Cross-lingual label-similarity panels")
plt.tight_layout()
plt.show()

plt.figure(figsize=(5, 3))
plt.plot([row["rung"] for row in results], [row["accuracy"] for row in results], marker="o")
plt.ylim(0, 1.05)
plt.ylabel("accuracy")
plt.xlabel("rung")
plt.title("Accuracy vs resource and coverage complexity")
plt.grid(True)
plt.show()


## Pitfall on D5: global score and norm drift hide language failure

A single global score can hide low-resource errors. Dot products can also reward vector norm instead of meaning, so we compare a broken dot-product classifier with cosine and per-language reporting.

In [ ]:

d5 = ladder[-1]["items"]
y_true = [item["label"] for item in d5]
langs = [item["language"] for item in d5]
cos_preds = []
dot_preds = []

for item in d5:
    emb = embed_text_multilingual(item["text"], item["language"])
    pred, scores, weights = crosslingual_classify(emb)
    cos_preds.append(pred)
    dot_scores = {label: float(np.dot(emb, vec * np.linalg.norm(vec))) for label, vec in LABEL_EMBS.items()}
    dot_preds.append(max(dot_scores, key=dot_scores.get))

cos_acc = accuracy_score(y_true, cos_preds)
dot_acc = accuracy_score(y_true, dot_preds)
per_language = {}
for language in sorted(set(langs)):
    mask = [lang == language for lang in langs]
    lang_true = [gold for gold, keep in zip(y_true, mask) if keep]
    lang_pred = [pred for pred, keep in zip(cos_preds, mask) if keep]
    per_language[language] = accuracy_score(lang_true, lang_pred)

print("cosine global accuracy", round(cos_acc, 3))
print("dot-product global accuracy", round(dot_acc, 3))
print("cosine per-language accuracy", per_language)
assert cos_acc >= dot_acc


## Evaluate it + Practice

- Metric: classification accuracy, sliced by language against a majority-label baseline.
- Sanity check: aligned English and Spanish vectors have diagonal similarity `0.9`.
- Ablation: replace cosine with dot product and inspect norm drift.
- Failure signals: high global score with low Swahili or Arabic accuracy, or sampling weights that quietly favor the wrong language.

Practice: change Arabic coverage from `0.6` to `0.9` and predict what should improve.

Practice: add a third neutral label vector and inspect zero-shot confusion.

Practice: try inverse-data-volume sampling and compare it with inverse-loss sampling.